# DTL: Feature Extraction and Generate Probability Layer using OVR method

# 1. Import satellite image from asset

In [ ]:
!python -m pip install .. --quiet

VERSION = 'v6'

import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')

# aoi definition

# import region from Hadi's assets
# region_name = "Sumatera"
# regions_fc = ee.FeatureCollection("users/hadicu06/IIASA/RESTORE/vector_datasets/classification_regions")
# aoi = regions_fc.filter(ee.Filter.eq('region_name', region_name)).geometry()

# try province export (aceh)

provinces = ee.FeatureCollection('projects/epistem2/assets/AOI_Sulawesi_Provinces')
province_list = provinces.toList(provinces.size())
aoi = ee.Feature(province_list.get(0)).geometry()

# Load satellite image stack from asset

stacked_landsat = ee.Image(f'projects/epistem2/assets/stacked_landsat_2020_sulawesi_{VERSION}')

# 2. Classification scheme 

In [ ]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

# 3. Load labelled training data from local

In [ ]:
import geopandas as gpd
from shapely.geometry import shape
import geemap

TrainVectPath = f"../data/modular_mapping_approach/sulawesi_test/sulawesi_td_DTL_result_{VERSION}.shp"
TrainData = gpd.read_file(TrainVectPath)
if TrainData.crs is None:
    TrainData = TrainData.set_crs("EPSG:4326")
aoi_geojson = aoi.getInfo()
aoi_geom = shape(aoi_geojson)
aoi_gdf = gpd.GeoDataFrame(
    {"geometry": [aoi_geom]},
    crs="EPSG:4326"
)
aoi_gdf = aoi_gdf.to_crs(TrainData.crs)

    
TrainDataFinal = gpd.clip(
    TrainData,
    aoi_gdf
)

print(f"Total training points : {len(TrainData)}")
print(f"Points inside AOI     : {len(TrainDataFinal)}")
print(f"Points removed        : {len(TrainData) - len(TrainDataFinal)}")
print(TrainData.columns.tolist())

# 4. Feature extraction from satellite imageries

In [ ]:
import geemap

# Settings

CLASS_PROPERTY = 'label'

N_TREES = 100
MIN_LEAF = 10
SEED = 0

band_names = stacked_landsat.bandNames()
labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

In [ ]:
# import geemap

# Map = geemap.Map()

# Map.centerObject(aoi, 7)
# Map.addLayer(aoi, {'color': 'red'}, 'AOI')
# Map.addLayer(stacked_landsat, {}, 'stacked_landsat_2020_Sumatera')
# Map.addLayer(labeled_roi, {}, 'labeled_roi')

# Map

In [ ]:
from luma_ge.classification import FeatureExtraction

feature_extractor = FeatureExtraction()

stratified_train, stratified_test = feature_extractor.stratified_split(
                                                    labeled_roi, 
                                                    stacked_landsat, 
                                                    class_prop=CLASS_PROPERTY, 
                                                    train_ratio=0.9
                                                )

# remove unecessary properties (i.e. primitives) from the training and testing datasets

input_props = band_names.add(CLASS_PROPERTY)
stratified_train = stratified_train.select(input_props)
stratified_test  = stratified_test.select(input_props)

# 5. Apply OVR classification

In [ ]:
from luma_ge.classification import Generate_LULC

classifier = Generate_LULC()


probability_stack = classifier.soft_classification(
    training_data=stratified_train,
    class_property=CLASS_PROPERTY,
    image=stacked_landsat,
    include_final_map=False,
    ntrees=N_TREES,
    v_split=None,
    min_leaf=MIN_LEAF,
    seed=SEED,
    probability_scale=100
)


In [ ]:
# comparison check: use multiprobability

# classifier_multiprob = ee.Classifier.smileRandomForest(
#     numberOfTrees=100,
#     minLeafPopulation=3,
#     bagFraction=0.5,
#     seed=0
# ).setOutputMode('MULTIPROBABILITY').train(
#     features=stratified_train,
#     classProperty=CLASS_PROPERTY,
#     inputProperties=stacked_landsat.bandNames()
# )

# probability_stack_multiprob = stacked_landsat.classify(classifier_multiprob)

# class_codes = ee.List(
#     stratified_train.aggregate_array(CLASS_PROPERTY)
# ).distinct().sort()

# class_labels = [
#     f"class_{code}"
#     for code in class_codes.getInfo()
# ]

# probability_stack_multiprob = probability_stack_multiprob.arrayFlatten(
#     [class_labels]
# )

In [ ]:
# sanity check

#1. Check number and names of probability bands
print("Bands:", probability_stack.bandNames().getInfo())
print("Number of bands:", probability_stack.bandNames().size().getInfo())

# 2. Check pixel type (should be Byte because of .byte())
print("Image type:", probability_stack.bandTypes().getInfo())

# 3. Check a single pixel only
sample = probability_stack.sample(
    region=aoi.centroid(),
    scale=aoi.projection().nominalScale(),
    numPixels=1,
    geometries=False
)

print("Single-pixel probability values:")
print(sample.first().getInfo())

In [ ]:
# Export the probability stack to an asset

# Export to Earth Engine Asset
# task = ee.batch.Export.image.toAsset(
#     image=probability_stack,
#     description='probability_stack_Sumsel_2020_td70',
#     assetId='projects/epistem2/assets/probability_stack_Sumsel_2020_td70',
#     region=aoi,
#     scale=100,
#     maxPixels=1e13
# )

# task.start()

# task = ee.batch.Export.image.toDrive(
#     image=probability_stack_multiprob,
#     description='probability_stack_multiprob_Aceh',
#     folder='GEE_Exports',
#     fileNamePrefix='probability_stack_multiprob_Aceh',
#     region=aoi,
#     scale=100,
#     maxPixels=1e13,
#     fileFormat='GeoTIFF'
# )

# task.start()